In [1]:
import numpy as np
import pandas as pd
import pyfpgrowth

In [2]:
csv_in = 'market_ex.csv'

In [3]:
df = pd.read_csv(csv_in, sep=',', skiprows=0, header=None)
print(df.shape)
print(df.info())
display(df.head())

(6008, 20)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6008 entries, 0 to 6007
Data columns (total 20 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       6008 non-null   object
 1   1       4592 non-null   object
 2   2       3491 non-null   object
 3   3       2674 non-null   object
 4   4       2029 non-null   object
 5   5       1497 non-null   object
 6   6       1106 non-null   object
 7   7       783 non-null    object
 8   8       524 non-null    object
 9   9       315 non-null    object
 10  10      202 non-null    object
 11  11      122 non-null    object
 12  12      71 non-null     object
 13  13      39 non-null     object
 14  14      20 non-null     object
 15  15      8 non-null      object
 16  16      4 non-null      object
 17  17      4 non-null      object
 18  18      3 non-null      object
 19  19      1 non-null      object
dtypes: object(20)
memory usage: 938.9+ KB
None


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,shrimp,almonds,avocado,vegetables mix,green grapes,whole weat flour,yams,cottage cheese,energy drink,tomato juice,low fat yogurt,green tea,honey,salad,mineral water,salmon,antioxydant juice,frozen smoothie,spinach,olive oil
1,burgers,meatballs,eggs,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,chutney,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,turkey,avocado,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,mineral water,milk,energy bar,whole wheat rice,green tea,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
df = df.replace(np.nan, '00nan')
display(df.head())

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,shrimp,almonds,avocado,vegetables mix,green grapes,whole weat flour,yams,cottage cheese,energy drink,tomato juice,low fat yogurt,green tea,honey,salad,mineral water,salmon,antioxydant juice,frozen smoothie,spinach,olive oil
1,burgers,meatballs,eggs,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan
2,chutney,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan
3,turkey,avocado,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan
4,mineral water,milk,energy bar,whole wheat rice,green tea,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan,00nan


In [5]:
ser_items = pd.Series(df.values.flatten())
top_items = ser_items.value_counts()
print(top_items.head())

00nan            96667
mineral water     1430
eggs              1117
spaghetti         1017
french fries      1009
Name: count, dtype: int64


In [6]:
id2item = sorted(list(set(df.values.flatten())))  # sort to fix order of items
print(len(id2item))  # debug
print(id2item[:5])  # debug
item2id = {}
for i in range(len(id2item)):
    item2id[id2item[i]] = i

120
['00nan', 'almonds', 'antioxydant juice', 'asparagus', 'avocado']


In [7]:
df_id = df.applymap(lambda x: item2id[x])
display(df_id.head())

C:\Users\iniad\AppData\Local\Temp\ipykernel_10708\3790243119.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_id = df.applymap(lambda x: item2id[x])


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,97,1,4,111,53,114,117,33,39,106,65,54,60,91,72,92,2,48,102,81
1,15,69,37,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,27,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,110,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,72,71,38,116,54,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [8]:
invoices = []
for i in range(df_id.shape[0]):
    ser = df_id.loc[i]
    s = ser[ ser>0 ]
    invoices.append(s)
print(len(invoices))
print(invoices[:5])

6008
[0      97
1       1
2       4
3     111
4      53
5     114
6     117
7      33
8      39
9     106
10     65
11     54
12     60
13     91
14     72
15     92
16      2
17     48
18    102
19     81
Name: 0, dtype: int64, 0    15
1    69
2    37
Name: 1, dtype: int64, 0    27
Name: 2, dtype: int64, 0    110
1      4
Name: 3, dtype: int64, 0     72
1     71
2     38
3    116
4     54
Name: 4, dtype: int64]


In [9]:
%time patterns = pyfpgrowth.find_frequent_patterns(invoices, 15)

CPU times: total: 453 ms
Wall time: 453 ms


In [10]:
%time rules = pyfpgrowth.generate_association_rules(patterns, 0.8)

CPU times: total: 15.6 ms
Wall time: 13.2 ms


In [11]:
print(rules)

{(76, 84): ((40,), 0.9444444444444444), (49, 55, 97): ((100,), 0.8095238095238095)}


In [12]:
results = []
for x in rules:
    ret = [x, rules[x][0], rules[x][1]]
    results.append(ret)
df_res = pd.DataFrame(results)
df_res.columns = ['LHS', 'RHS', 'Conf']

In [13]:
display(df_res.sort_values(by='Conf', ascending=False))

,LHS,RHS,Conf
0,"(76, 84)","(40,)",0.944444
1,"(49, 55, 97)","(100,)",0.809524


In [14]:
print(id2item[76])
print(id2item[84])
print(id2item[40])

mushroom cream sauce
pasta
escalope


In [15]:
n_all = len(invoices)
lift = []
for i in range(df_res.shape[0]):
    rhs = df_res.at[i, 'RHS']
    conf = df_res.at[i, 'Conf']
    n_rhs = 0
    for items in invoices:
        if set(items) >= set(rhs):
            n_rhs += 1
    lift1 = conf / (n_rhs / n_all)
    lift.append(lift1)
    
df_res['Lift'] = lift

In [16]:
display(df_res.sort_values(by='Conf', ascending=False))

,LHS,RHS,Conf,Lift
0,"(76, 84)","(40,)",0.944444,11.72360
1,"(49, 55, 97)","(100,)",0.809524,4.78232
